# Music4All: full-catalog recommendation training
Read `docs/KAGGLE_TRAINING.md` first. Original Music4All must be obtained from its authors; keep data and notebook private according to your access terms. This notebook does not download restricted audio or mount Drive automatically.
Default: full original inventory (109,269 tracks), global temporal split, ALS + genre + popularity. No implicit sampling or fallback split.

In [ ]:
import sys, subprocess
from pathlib import Path
assert (3, 10) <= sys.version_info[:2] < (3, 13), 'Use a Python 3.10–3.12 kernel for the pinned environment'
# Add the private code Dataset made by scripts/package_kaggle_code.py.
matches = list(Path('/kaggle/input').rglob('requirements-kaggle.txt'))
assert len(matches) == 1, 'Attach exactly one copy of the music4all training code Dataset'
CODE = matches[0].parent
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(CODE / 'requirements-kaggle.txt')], check=True)
print('Dependencies installed. If Kaggle requests a kernel restart, restart before continuing.')

In [ ]:
import json, os
from pathlib import Path
# EDIT these paths after inspecting the attached private Dataset folders.
ORIGINAL = Path('/kaggle/input/music4all-original/music4all')
ONION = Path('/kaggle/input/music4all-onion')
RUN = Path('/kaggle/working/music4all-run')
CODE = next(Path('/kaggle/input').rglob('requirements-kaggle.txt')).parent
config = json.loads((CODE / 'configs/full.json').read_text())
config.update(original_root=str(ORIGINAL), onion_root=str(ONION), output_root=str(RUN), scratch_root='/tmp/music4all-training')
# CPU ALS is the tested default. GPU requires a compatible implicit CUDA runtime.
config['use_gpu'] = False
config['max_users'] = None  # full source users who have training history
assert ORIGINAL.is_dir(), 'Attach the extracted complete original Music4All Dataset'
assert (ONION / 'userid_trackid_timestamp.tsv.bz2').is_file(), 'Attach timestamp file; no silent hash fallback'
assert not RUN.exists(), 'Use a fresh RUN directory for each experiment'
CONFIG = Path('/kaggle/working/training-config.json')
CONFIG.write_text(json.dumps(config, indent=2))
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = str(config['threads'])
print('Full-catalog run configured; no source audio will be copied to notebook output.')

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, str(CODE / 'validate_original.py'), '--root', str(ORIGINAL), '--report', '/kaggle/working/original-inventory.json'], check=True)
subprocess.run([sys.executable, str(CODE / 'prepare.py'), '--config', str(CONFIG)], check=True)

In [ ]:
subprocess.run([sys.executable, str(CODE / 'train.py'), '--config', str(CONFIG)], check=True)
metrics = json.loads((RUN / 'metrics.json').read_text())
print(json.dumps(metrics, indent=2))

In [ ]:
users = json.loads((RUN / 'prepared/user_ids.json').read_text())
subprocess.run([sys.executable, str(CODE / 'predict.py'), '--run', str(RUN), '--user-id', users[0], '--limit', '10'], check=True)
# Cold start by explicit genre (must be a name listed in genre_names.json):
genres = json.loads((RUN / 'prepared/genre_names.json').read_text())
subprocess.run([sys.executable, str(CODE / 'predict.py'), '--run', str(RUN), '--genre', genres[0], '--limit', '10'], check=True)

## Optional: separate genre classification experiment
This predicts dataset genre labels from audio features; it does not replace the recommender. Attach a numerical ID-aligned audio feature TSV (e.g. `id_musicnn.tsv.bz2`). Do not pass genre/tag-derived features as input. The script validates its header and reports omitted rare labels.

In [ ]:
RUN_CLASSIFIER = False
if RUN_CLASSIFIER:
    audio_features = ONION / 'id_musicnn.tsv.bz2'
    assert audio_features.is_file()
    subprocess.run([sys.executable, str(CODE / 'train_genre_classifier.py'), '--prepared', str(RUN / 'prepared'), '--audio-features', str(audio_features), '--output', str(RUN / 'genre-classifier')], check=True)

## Save results
Use **Save Version → Save & Run All** and retain outputs. Download the run folder or save it as a private Dataset. Do not copy raw audio or scratch DuckDB into `/kaggle/working`.
Artifacts are offline-training outputs. The existing FastAPI demo still uses its SQLite Item-kNN artifact; importing ALS into the serving API is a separate integration step.
Full audio archive lives in your private source storage. Only numerical features needed by the selected experiment must be read during training; having the full archive does not require fitting a deep audio encoder from scratch.

In [ ]:
size = sum(p.stat().st_size for p in RUN.rglob('*') if p.is_file())
print(f'Run artifact size: {size / 1024**3:.2f} GiB')
print('Download / save the complete run directory, including config, model, prepared mappings and matrices, metrics and artifact_manifest.json.')
assert (RUN / 'artifact_manifest.json').is_file()